# GHCNd: quickstart

Download and plot daily temperature and precipitation for London Heathrow
from the Global Historical Climatology Network - daily.

See [`docs/ghcnd/README.md`](../docs/ghcnd/README.md) for the full
reference. No authentication. `pip install pandas requests matplotlib`.

In [ ]:
# ==================================================================
# USER CONFIGURATION - Edit these values for your use case
# ==================================================================
STATION_ID = "UKM00003772"  # London Heathrow
ELEMENTS = ("TMAX", "TMIN", "PRCP")
OUTPUT_DIR = "../data/ghcnd"
OUTPUT_FILENAME = "ghcnd_heathrow.csv"
# ==================================================================

## Imports and environment check

In [ ]:
import sys
from importlib.metadata import version
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

print(f"Python       {sys.version.split()[0]}")
for pkg in ["pandas", "matplotlib", "requests"]:
    print(f"{pkg:12} {version(pkg)}")

def _find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for parent in [here, *here.parents]:
        if (parent / "CLAUDE.md").exists():
            return parent
    raise RuntimeError("Could not find repo root.")

repo_root = _find_repo_root()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

## Download and parse

GHCNd `.dly` files are fixed-width ASCII with 31 daily value columns per
row (one row per station/year/month/element). The helper in
`scripts/ghcnd_download.py` parses this into a tidy long-format CSV,
scales TMAX/TMIN/PRCP from tenths to natural units, and drops
QC-failing rows.

In [ ]:
from scripts.ghcnd_download import download  # noqa: E402

output_path = download(
    station_id=STATION_ID,
    elements=ELEMENTS,
    output_dir=OUTPUT_DIR,
    output_filename=OUTPUT_FILENAME,
)
print(f"Saved: {output_path}")

## Open and inspect

One row per (date, element). TMAX and TMIN are in degC, PRCP in mm.

In [ ]:
df = pd.read_csv(output_path, parse_dates=["date"])
print(df.head(10))
print(f"\nRows: {len(df):,}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Elements present: {sorted(df['element'].unique())}")

## Plot daily TMAX and TMIN for recent years

Pivot to one column per element, restrict to the last 5 years for
readability, and plot the daily envelope.

In [ ]:
wide = df.pivot_table(index="date", columns="element", values="value")
wide = wide.loc[wide.index >= wide.index.max() - pd.Timedelta(days=5 * 365)]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
if "TMAX" in wide.columns and "TMIN" in wide.columns:
    ax1.fill_between(wide.index, wide["TMIN"], wide["TMAX"], color="C0", alpha=0.3, label="Daily range")
    ax1.plot(wide.index, wide["TMAX"], color="C3", linewidth=0.5, label="TMAX")
    ax1.plot(wide.index, wide["TMIN"], color="C0", linewidth=0.5, label="TMIN")
    ax1.set_ylabel("Temperature (C)")
    ax1.legend(loc="upper left")
    ax1.grid(alpha=0.3)
    ax1.set_title(f"Heathrow daily temperature, last 5 years (GHCNd station {STATION_ID})")

if "PRCP" in wide.columns:
    ax2.bar(wide.index, wide["PRCP"], color="C2", width=1.0, label="PRCP")
    ax2.set_ylabel("Precipitation (mm)")
    ax2.grid(alpha=0.3)
    ax2.set_xlabel("Date")

plt.tight_layout()
plt.show()

## Next steps

- **Full reference**: [`docs/ghcnd/README.md`](../docs/ghcnd/README.md)
- **Find more UK stations**: download `ghcnd-stations.txt`, filter by the
  `UK` prefix, and explore nearby sites by latitude/longitude.
- **Compare multiple stations**: loop over station IDs, concatenate the
  resulting DataFrames with a `station` column.
- **Derive heatwave indices**: e.g. days per year with TMAX above a
  percentile threshold; GHCNd station records give you real observations
  to anchor these against.
- **Compare with ERA5**: pull ERA5 `2m_temperature` at the same grid cell
  and overlay against this station's daily values to see how the
  reanalysis reproduces point observations.